In [1]:
#import

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

#Load the secrets
load_dotenv("secret.env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
MINIO_BUCKET_BRONZE = os.getenv("MINIO_BUCKET_BRONZE")

#Spark session creation
spark = SparkSession.builder \
    .appName("Bank_data_pipeline") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.2"
    ) \
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        f"http://{MINIO_ENDPOINT}"
    ) \
    .config(
        "spark.hadoop.fs.s3a.access.key",
        f"{MINIO_ACCESS_KEY}"
    ) \
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        f"{MINIO_SECRET_KEY}"
    ) \
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    ) \
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    ) \
    .getOrCreate()


In [2]:
# Defining the Schema

schema = StructType([
    StructField("step",IntegerType()),
    StructField("type",StringType()),
    StructField("amount",DoubleType()),
    StructField("Account_ID",StringType()),
    StructField("old_balance",DoubleType()),
    StructField("new_balance",DoubleType()),
    StructField("receiver_ID",StringType()),
    StructField("receiver_old_bal",DoubleType()),
    StructField("receiver_new_bal",DoubleType()),
    StructField("is_fraud",IntegerType())
])

In [3]:
# Reading the data & checking the schema of data

df = spark.read.format("csv")\
               .option("header", True)\
               .schema(schema)\
               .option("mode","PERMISSIVE")\
               .option("path","/home/jupyter/Data_Engineer_project/Dataset/transaction_bank_data.csv")\
               .load()

df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- Account_ID: string (nullable = true)
 |-- old_balance: double (nullable = true)
 |-- new_balance: double (nullable = true)
 |-- receiver_ID: string (nullable = true)
 |-- receiver_old_bal: double (nullable = true)
 |-- receiver_new_bal: double (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [4]:
#Displaying sample 10 data's

df.show(10, truncate= False)

+----+--------+--------+-----------+-----------+-----------+-----------+----------------+----------------+--------+
|step|type    |amount  |Account_ID |old_balance|new_balance|receiver_ID|receiver_old_bal|receiver_new_bal|is_fraud|
+----+--------+--------+-----------+-----------+-----------+-----------+----------------+----------------+--------+
|1   |PAYMENT |9839.64 |C1231006815|170136.0   |160296.36  |M1979787155|0.0             |0.0             |0       |
|1   |PAYMENT |1864.28 |C1666544295|21249.0    |19384.72   |M2044282225|0.0             |0.0             |0       |
|1   |TRANSFER|181.0   |C1305486145|181.0      |0.0        |C553264065 |0.0             |0.0             |1       |
|1   |CASH_OUT|181.0   |C840083671 |181.0      |0.0        |C38997010  |21182.0         |0.0             |1       |
|1   |PAYMENT |11668.14|C2048537720|41554.0    |29885.86   |M1230701703|0.0             |0.0             |0       |
|1   |PAYMENT |7817.71 |C90045638  |53860.0    |46042.29   |M573487274 |

In [5]:
#Total row count
df.count()

6362620

In [6]:
# Total partition used
df.rdd.getNumPartitions()

8

In [7]:
#Creating Load date by processing the Step column

df_raw = df.withColumn("load_date", date_add(to_date(lit('2026-01-01')), 
                                             (((floor((col("step") - 1) / 24)) + 1) - 1).cast("int")))

In [8]:
# Upload to minio (S3 compatible)

from minio import Minio

client = Minio(
    f"{MINIO_ENDPOINT}",
    access_key=f"{MINIO_ACCESS_KEY}",
    secret_key=f"{MINIO_SECRET_KEY}",
    secure=False
)


In [9]:
# writing the data with Load_date as Partition in Bronze layer

cutoff_date = '2026-01-20'

df_raw.filter(col("load_date")<=cutoff_date)\
      .write.mode("overwrite")\
      .format("parquet")\
      .partitionBy("load_date")\
      .save(f"s3a://{MINIO_BUCKET_BRONZE}/transactions_batch/")

In [10]:
# checking the bronze layer data

df_bronze = spark.read.format("parquet")\
                      .option("path",f"s3a://{MINIO_BUCKET_BRONZE}/transactions_batch/")\
                      .load()

df_bronze.show()

+----+--------+----------+-----------+-----------+-----------+-----------+----------------+----------------+--------+----------+
|step|    type|    amount| Account_ID|old_balance|new_balance|receiver_ID|receiver_old_bal|receiver_new_bal|is_fraud| load_date|
+----+--------+----------+-----------+-----------+-----------+-----------+----------------+----------------+--------+----------+
|   1| PAYMENT|   9839.64|C1231006815|   170136.0|  160296.36|M1979787155|             0.0|             0.0|       0|2026-01-01|
|  15| PAYMENT|  21042.07| C641303564|     8165.0|        0.0|M1535596143|             0.0|             0.0|       0|2026-01-01|
|   1| PAYMENT|   1864.28|C1666544295|    21249.0|   19384.72|M2044282225|             0.0|             0.0|       0|2026-01-01|
|  15|CASH_OUT|  40914.95| C134768139|     1071.0|        0.0|C1803962165|          4270.0|      6583732.39|       0|2026-01-01|
|   1|TRANSFER|     181.0|C1305486145|      181.0|        0.0| C553264065|             0.0|      